In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark import StorageLevel
import pandas as pd
import time

In [3]:
spark = SparkSession.builder \
    .appName("NYC Taxi Task4") \
    .getOrCreate()

In [4]:
df = spark.read.parquet("yellow_tripdata_2024-01.parquet")

Check Initial Partitions

In [5]:
print("Initial Partitions:")
print(df.rdd.getNumPartitions())

Initial Partitions:
2


Benchmark 1 - Measure Execution Time Without Cache

In [6]:
start = time.time()

df.groupBy("payment_type") \
  .agg(avg("fare_amount")) \
  .show()

without_cache_time = time.time() - start

print("Without Cache:", without_cache_time)

+------------+------------------+
|payment_type|  avg(fare_amount)|
+------------+------------------+
|           1|18.557432202727387|
|           3| 6.752568760524564|
|           2|17.866037304953625|
|           4|1.3348886934888904|
|           0|20.016193904200065|
+------------+------------------+

Without Cache: 4.131690740585327


Apply Cache

In [7]:
df.cache()

df.count()

2964624

Benchmark 2 - Measure Execution Time With Cache

In [8]:
start = time.time()

df.groupBy("payment_type") \
  .agg(avg("fare_amount")) \
  .show()

with_cache_time = time.time() - start

print("With Cache:", with_cache_time)

+------------+------------------+
|payment_type|  avg(fare_amount)|
+------------+------------------+
|           1|18.557432202727387|
|           3| 6.752568760524564|
|           2|17.866037304953625|
|           4|1.3348886934888904|
|           0|20.016193904200065|
+------------+------------------+

With Cache: 2.8527700901031494


Compare Cache Performance

In [9]:
print("Without Cache:", without_cache_time)

print("With Cache:", with_cache_time)

improvement = (
    (without_cache_time - with_cache_time)
    /
    without_cache_time
) * 100

print("Improvement %:", improvement)

Without Cache: 4.131690740585327
With Cache: 2.8527700901031494
Improvement %: 30.953929777933865


Persist Dataset

In [10]:
from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)

df.count()

print("Dataset Persisted")

Dataset Persisted


Repartition Dataset

In [11]:
repartitioned_df = df.repartition(4)

print(
    "Partitions After Repartition:"
)

print(
    repartitioned_df.rdd.getNumPartitions()
)

Partitions After Repartition:
4


Benchmark 3 - Measure Performance After Repartition

In [12]:
start = time.time()

repartitioned_df.groupBy(
    "payment_type"
).agg(
    avg("tip_amount")
).show()

repartition_time = time.time() - start

print(
    "Execution Time:",
    repartition_time
)

+------------+--------------------+
|payment_type|     avg(tip_amount)|
+------------+--------------------+
|           0|   1.545956750046391|
|           1|   4.169670627492557|
|           3|0.014559881614532841|
|           2|0.002296016994883...|
|           4| 0.04212511795487691|
+------------+--------------------+

Execution Time: 6.785818099975586


Spark Configuration

In [13]:
spark.sparkContext.getConf().getAll()

[('spark.rdd.compress', 'True'),
 ('spark.driver.port', '37805'),
 ('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K'),
 ('spark.app.name', 'NYC Taxi Task4'),
 ('spark.driver.host', 'db7752c84dad'),
 ('spark.executor.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.

Application Details

In [14]:
print(
    "Application ID:",
    spark.sparkContext.applicationId
)

print(
    "Spark Version:",
    spark.version
)

Application ID: local-1782615300308
Spark Version: 4.0.3


Scalability Table

In [15]:
scalability_df = pd.DataFrame(
    {
        "Metric":[
            "Without Cache",
            "With Cache",
            "Repartitioned"
        ],
        "Execution_Time":[
            without_cache_time,
            with_cache_time,
            repartition_time
        ]
    }
)

scalability_df

,Metric,Execution_Time
0,Without Cache,4.131691
1,With Cache,2.852770
2,Repartitioned,6.785818


Export For Tableau

In [16]:
scalability_df.to_csv(
    "scalability_metrics.csv",
    index=False
)

print(
    "Scalability Metrics Saved"
)

Scalability Metrics Saved


In [17]:
# Install ngrok
!pip install -q pyngrok

In [19]:
from pyngrok import ngrok

ngrok.set_auth_token("3FkOG3Cz6FuwaK88Jv0umWiE02y_63q1FtB7hEuED37f4GjxN")

In [21]:
# Before Benchmark 1 (Without Cache)

spark.sparkContext.setJobDescription(
    "Benchmark 1 - Aggregation Without Cache"
)

start = time.time()

df.groupBy(
    "payment_type"
).agg(
    avg("fare_amount")
).show()

+------------+------------------+
|payment_type|  avg(fare_amount)|
+------------+------------------+
|           1|18.557432202727387|
|           3| 6.752568760524564|
|           2|17.866037304953625|
|           4|1.3348886934888904|
|           0|20.016193904200065|
+------------+------------------+



In [22]:
spark.sparkContext.setJobDescription(
    "Materialize Cache"
)

df.cache()

df.count()

2964624

In [23]:
# Before Persist

spark.sparkContext.setJobDescription(
    "Persist Dataset (MEMORY_AND_DISK)"
)

from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)

df.count()

2964624

In [24]:
# Before Benchmark 2 (With Cache)

spark.sparkContext.setJobDescription(
    "Benchmark 2 - Aggregation With Cache"
)

start = time.time()

df.groupBy(
    "payment_type"
).agg(
    avg("fare_amount")
).show()

+------------+------------------+
|payment_type|  avg(fare_amount)|
+------------+------------------+
|           1|18.557432202727387|
|           3| 6.752568760524564|
|           2|17.866037304953625|
|           4|1.3348886934888904|
|           0|20.016193904200065|
+------------+------------------+



In [25]:
# Before Repartition

spark.sparkContext.setJobDescription(
    "Repartition Dataset"
)

repartitioned_df = df.repartition(4)

In [26]:
# Before Benchmark 3

spark.sparkContext.setJobDescription(
    "Benchmark 3 - Repartition Performance"
)

start = time.time()

repartitioned_df.groupBy(
    "payment_type"
).agg(
    avg("tip_amount")
).show()

+------------+--------------------+
|payment_type|     avg(tip_amount)|
+------------+--------------------+
|           0|   1.545956750046391|
|           1|   4.169670627492557|
|           3|0.014559881614532841|
|           2|0.002296016994883...|
|           4| 0.04212511795487691|
+------------+--------------------+



In [27]:
public_url = ngrok.connect(4040)
print(public_url)

NgrokTunnel: "https://festival-cranberry-purposely.ngrok-free.dev" -> "http://localhost:4040"


Resource Configuration

In [31]:
from pyspark.sql import SparkSession

conf = spark.sparkContext.getConf()

print("="*65)
print("RESOURCE CONFIGURATION")
print("="*65)

print(f"{'Spark Version':35} = {spark.version}")
print(f"{'Application Name':35} = {spark.sparkContext.appName}")
print(f"{'Master':35} = {spark.sparkContext.master}")
print(f"{'Application ID':35} = {spark.sparkContext.applicationId}")
print(f"{'Default Parallelism':35} = {spark.sparkContext.defaultParallelism}")
print(f"{'Initial Partitions':35} = {df.rdd.getNumPartitions()}")

print(f"{'Driver Memory':35} = {conf.get('spark.driver.memory', 'Default (Managed by Colab)')}")
print(f"{'Executor Memory':35} = {conf.get('spark.executor.memory', 'Default (Local Mode)')}")
print(f"{'Shuffle Partitions':35} = {conf.get('spark.sql.shuffle.partitions', '200 (Default)')}")
print(f"{'Serializer':35} = {conf.get('spark.serializer', 'JavaSerializer (Default)')}")

RESOURCE CONFIGURATION
Spark Version                       = 4.0.3
Application Name                    = NYC Taxi Task4
Master                              = local[*]
Application ID                      = local-1782615300308
Default Parallelism                 = 2
Initial Partitions                  = 2
Driver Memory                       = Default (Managed by Colab)
Executor Memory                     = Default (Local Mode)
Shuffle Partitions                  = 200 (Default)
Serializer                          = JavaSerializer (Default)
